### Feature Fusion

In [1]:
import pandas as pd

deep_df = pd.read_csv("../Outputs/deep_features.csv")

print("--- Member A's Column Names ---")
# Print the first 10 column names to see what the image column is called
print(deep_df.columns.tolist()[:10])

--- Member A's Column Names ---
['image_id', 'split', 'label', 'label_name', 'pred_label', 'pred_label_name', 'prob_malignant', 'deep_0000', 'deep_0001', 'deep_0002']


In [2]:
import os
import pandas as pd

radiomics_path = "../Outputs/radiomics_features.csv"
deep_path = "../Outputs/deep_features.csv" 
fusion_output_path = "../Outputs/fusion_dataset.csv"

print("Loading real feature tables for alignment...")
radiomics_df = pd.read_csv(radiomics_path)
deep_df = pd.read_csv(deep_path)

# --- THE FIX: We are explicitly keeping the 'split' column now! ---
deep_feature_cols = [col for col in deep_df.columns if col.startswith('deep_')]
cols_to_keep = ['image_id', 'split'] + deep_feature_cols

deep_df_filtered = deep_df[cols_to_keep].copy() 
print(f"Cleaned Deep Features shape: {deep_df_filtered.shape}")

# Create temporary integer column to fix the string/int mismatch
radiomics_df['temp_match_id'] = radiomics_df['image_name'].str.replace('.jpg', '', regex=False).astype(int)

# Merge
fusion_df = pd.merge(deep_df_filtered, radiomics_df, left_on="image_id", right_on="temp_match_id")

# Clean up duplicate ID columns
fusion_df = fusion_df.drop(columns=["image_id", "temp_match_id"])

# Save
fusion_df.to_csv(fusion_output_path, index=False)

print("\n--- FEATURE FUSION VERIFICATION ---")
print(f"Fused dataset saved. 'split' column preserved: {'split' in fusion_df.columns}")
print(f"Total columns: {fusion_df.shape[1]}")

Loading real feature tables for alignment...
Cleaned Deep Features shape: (5000, 1282)

--- FEATURE FUSION VERIFICATION ---
Fused dataset saved. 'split' column preserved: True
Total columns: 1301


### Sanity check 

In [5]:
import pandas as pd

# Load your newly fused dataset
fusion_df = pd.read_csv("../Outputs/fusion_dataset.csv")

print("--- FUSION INTEGRITY CHECKS ---")

# 1. THE NULL CHECK (The most important test)
# If Pandas failed to match an ID, it doesn't always crash; it often just fills the row with 'NaN' (Not a Number).
total_nans = fusion_df.isna().sum().sum()
print(f"1. Total missing values (NaNs): {total_nans}")
if total_nans == 0:
    print("   ✅ Perfect! No missing data from mismatched rows.")
else:
    print("   ❌ WARNING: Missing data detected. The translation bridge failed on some images.")

# 2. THE COLUMN CHECK
# Let's verify we have the exact ingredients expected for the neural network.
deep_count = sum(col.startswith('deep_') for col in fusion_df.columns)
glcm_count = sum(col.startswith('glcm_') for col in fusion_df.columns)
lbp_count = sum(col.startswith('lbp_') for col in fusion_df.columns)
shape_count = sum(col.startswith('shape_') for col in fusion_df.columns)

print("\n2. Verifying feature representation:")
print(f"   Image Name column exists: {'image_name' in fusion_df.columns}")
print(f"   Label column exists: {'label' in fusion_df.columns}")
print(f"   Deep features: {deep_count} (Expected: 1280)")
print(f"   Radiomics features: {glcm_count + lbp_count + shape_count} (Expected: 18)")

# 3. THE VISUAL INSPECTION
# Let's look at a slice of the first row to make sure the numbers actually transferred properly side-by-side.
print("\n3. First row preview (Image Name + First 2 Deep + First 2 GLCM):")
preview_cols = ['image_name'] + [c for c in fusion_df.columns if c.startswith('deep_')][:2] + [c for c in fusion_df.columns if c.startswith('glcm_')][:2]
print(fusion_df[preview_cols].head(1))

--- FUSION INTEGRITY CHECKS ---
1. Total missing values (NaNs): 0
   ✅ Perfect! No missing data from mismatched rows.

2. Verifying feature representation:
   Image Name column exists: True
   Label column exists: True
   Deep features: 1280 (Expected: 1280)
   Radiomics features: 18 (Expected: 18)

3. First row preview (Image Name + First 2 Deep + First 2 GLCM):
   image_name  deep_0000  deep_0001  glcm_contrast  glcm_correlation
0  000002.jpg   0.587635   0.192216      33.644147          0.992533
